# Mašinsko učenje — Vežba 1 — Zadatak 2
## Implementacija polinomijalne regresije u scikit-learn

Skup podataka ima 10 parova: godišnje padavine $X$ (mm) i biljna biomasa $Y$ (t/ha).
Pravimo klasu `PolynomialRegression`, izvedenu iz `BaseEstimator` i `RegressorMixin`, sa metodama `fit`, `predict` i `score`.

Matrični oblik normalnih jednačina:

$$ (\Phi^T\Phi)\mathbf w = \Phi^T\mathbf y, \qquad \Phi_{ij}=x_i^j. $$

U zadatku se `score` definiše kao **MSE**, a ne kao uobičajeni $R^2$.

In [1]:
import pandas as pd
import numpy as np
from poly_regression import PolynomialRegression

podaci = pd.read_csv('podaci.csv')
print('Ucitano merenja:', len(podaci))
display(podaci)

Ucitano merenja: 10


,X,Y
0,520,3.1
1,610,3.8
2,700,4.0
3,820,5.1
4,910,5.3
5,1030,6.0
6,1120,6.2
7,1250,5.9
8,1390,5.1
9,1510,4.1


### Obuka modela

Odabran je polinom drugog stepena, $\hat y=w_0+w_1x+w_2x^2$.
`fit` formira matricu $\Phi$ i računa koeficijente, a `predict` vraća procenjenu biomasu.

In [2]:
X = podaci[['X']].to_numpy()
y = podaci['Y'].to_numpy()

model = PolynomialRegression(degree=2)
model.fit(X, y)

y_pred = model.predict(X)
rezultat = podaci.copy()
rezultat['Y_pred'] = y_pred
rezultat['rezidual'] = rezultat['Y'] - rezultat['Y_pred']

display(rezultat.round(6))
print('w =', model.coef_)
print(f'Err = {model.score(X, y) / 2:.9f}')
print(f'MSE = {model.score(X, y):.9f}')
print(f'Predvidjena biomasa pri 900 mm: {model.predict([[900]])[0]:.6f} t/ha')

,X,Y,Y_pred,rezidual
0,520,3.1,2.817169,0.282831
1,610,3.8,3.707915,0.092085
2,700,4.0,4.447484,-0.447484
3,820,5.1,5.198413,-0.098413
4,910,5.3,5.585237,-0.285237
5,1030,6.0,5.865840,0.134160
6,1120,6.2,5.899919,0.300081
7,1250,5.9,5.682253,0.217747
8,1390,5.1,5.095099,0.004901
9,1510,4.1,4.300670,-0.200670


w = [-5.28943056e+00  2.04421901e-02 -9.33187438e-06]
Err = 0.028775682
MSE = 0.057551364
Predvidjena biomasa pri 900 mm: 5.549722 t/ha


### Kontrola rezultata

Za $M=2$ očekuje se $\mathrm{MSE}\approx 0.057551364$. Funkcija greške iz prvog zadatka je $Err=\mathrm{MSE}/2$.

In [3]:
assert np.isclose(model.score(X, y), 0.05755136383219559, rtol=1e-7)
print('Provera je uspesna.')

Provera je uspesna.


### Objašnjenje metoda

- `__init__`: čuva stepen polinoma i koeficijent regularizacije, podrazumevano nula.
- `fit`: konstruiše Vandermondeovu matricu i rešava normalne jednačine.
- `predict`: računa vrednosti polinoma za prosleđeno $X$.
- `score`: računa $\frac1N\sum_i(y_i-\hat y_i)^2$.

Definicija klase nalazi se u `poly_regression.py`, a samostalni test u `test_poly_regression.py`.